## Predicting Floods in Sub-Saharan Africa Using Satellite Imagery and Computer Vision

# Project Objective

Developing a machine learning system that can predict floods 3 days in advance in Sub-Saharan Africa using Satellite imagery and related environmental data. The focus will be on demonstrating the application of computer vision techniques to extract spatial patterns predictive of flooding


# Motivation

- Floods are a major hazard in Sub-Saharan Africa, causing property damage, displacement, and loss of life.
- Early prediction enables disaster preparedness and humanitarian response.
- Satellite imagery is increasingly available, offering a rich visual dataset for flood prediction.
- This project demonstrates the use of computer vision pipelines in a real-world social-impact context.


In [1]:
# Load necessary libraries

# Core
import os
import glob
from pathlib import Path

import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import geemap

# Geospatial + raster
import rasterio
from rasterio.plot import show
import geopandas as gpd
import ee
# ml4floods not installed in this env — avoid importing it here
# from ml4floods import ...
# from ml4floods.data import worldfloods
# georeader not installed in this env — avoid importing window_utils
# from georeader import window_utils
from shapely.geometry import box


# Notebook display
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Local fallback for band ordering (used by Dataset). Adjust if your tiles use a different order.
BANDS_S2 = ['B1','B2','B3','B4','B5','B6','B7','B8','B8A','B9','B10','B11','B12']

print("Imports loaded.")


Imports loaded.


## Standard U-Net

The Architecture: Vanilla U-Net
- Backbone: ResNet-18 or ResNet-34 encoder as it is light to train on my resource constrained computer
- Loss Function: Dice Loss or combination of Binary Cross Entropy + Dice to handle the class imbalance since most of the image is land and a small part is flood

In [2]:
# ResNet-backed U-Net (lightweight; supports resnet18 or resnet34 encoder)
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models


class ConvRelu(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, padding=1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=kernel_size, padding=padding)
        self.bn = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        # Upsample then convs after concatenation with skip
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv1 = ConvRelu(in_ch + skip_ch, out_ch)
        self.conv2 = ConvRelu(out_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        # If necessary, pad to match skip size
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return x


class UNetResNet(nn.Module):
    """
    U-Net with a ResNet encoder (choose depth=18 or 34). Outputs logits (no sigmoid) for use with BCEWithLogitsLoss.
    """
    def __init__(self, encoder='resnet18', pretrained=False, out_channels=1, n_input_channels=3):
        super().__init__()
        assert encoder in ('resnet18', 'resnet34')
        if encoder == 'resnet18':
            resnet = models.resnet18(pretrained=pretrained)
            filters = [64, 64, 128, 256, 512]
        else:
            resnet = models.resnet34(pretrained=pretrained)
            filters = [64, 64, 128, 256, 512]

        # Adapt first conv if input channels != 3
        if n_input_channels != 3:
            old_conv = resnet.conv1
            resnet.conv1 = nn.Conv2d(n_input_channels, old_conv.out_channels,
                                     kernel_size=old_conv.kernel_size,
                                     stride=old_conv.stride,
                                     padding=old_conv.padding,
                                     bias=old_conv.bias)

        # Encoder (take sequential layers from ResNet)
        self.initial = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)  # produces 64 ch
        self.pool = resnet.maxpool
        self.encoder1 = resnet.layer1  # 64
        self.encoder2 = resnet.layer2  # 128
        self.encoder3 = resnet.layer3  # 256
        self.encoder4 = resnet.layer4  # 512

        # Center / bridge
        self.center = nn.Sequential(
            ConvRelu(filters[4], filters[4]),
            ConvRelu(filters[4], filters[4])
        )

        # Decoder blocks (match channels to encoder skip connections)
        self.dec4 = DecoderBlock(filters[4], filters[3], filters[3])  # 512 -> +256 -> 256
        self.dec3 = DecoderBlock(filters[3], filters[2], filters[2])  # 256 -> +128 -> 128
        self.dec2 = DecoderBlock(filters[2], filters[1], filters[1])  # 128 -> +64 -> 64
        self.dec1 = DecoderBlock(filters[1], filters[0], filters[0])  # 64 -> +64 -> 64

        # Final conv to output desired channels
        self.final_conv = nn.Sequential(
            ConvRelu(filters[0], filters[0] // 2),
            nn.Conv2d(filters[0] // 2, out_channels, kernel_size=1)
        )

    def forward(self, x):
        # Encoder
        x0 = self.initial(x)      # after conv1: 64 ch, spatial /2
        x1 = self.pool(x0)        # /4
        e1 = self.encoder1(x1)    # 64
        e2 = self.encoder2(e1)    # 128
        e3 = self.encoder3(e2)    # 256
        e4 = self.encoder4(e3)    # 512

        # Center
        c = self.center(e4)

        # Decoder (upsample + skip connections)
        d4 = self.dec4(c, e3)
        d3 = self.dec3(d4, e2)
        d2 = self.dec2(d3, e1)
        d1 = self.dec1(d2, x0)

        out = self.final_conv(d1)
        # returns logits; apply sigmoid in training/inference as needed
        return out

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Dataset, DataLoader, training and evaluation for S2 -> GT flood masks
import random
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
# use local BANDS_S2 defined in the imports cell above instead of importing from ml4floods


class WorldFloodsTilesDataset(Dataset):
    """Reads S2 tiles and aligned GT masks.
    - bands: list of band names, e.g. ['B4','B3','B8']
    - max_samples: optional cap for quick tests
    - transforms: simple callable(image, mask) -> (image, mask)
    """
    def __init__(self, data_root, split='train', bands=('B4','B3','B8'), max_samples=None, transforms=None, scale=3500.0):
        self.data_root = Path(data_root)
        self.s2_dir = self.data_root / split / 'S2'
        self.gt_dir = self.data_root / split / 'gt'
        self.transforms = transforms
        self.scale = float(scale)
        self.band_names = list(bands)
        # map band names to 1-based rasterio indexes using local BANDS_S2
        self.band_idxs = [BANDS_S2.index(b) + 1 for b in self.band_names]

        s2_files = sorted(self.s2_dir.glob('*.tif')) if self.s2_dir.exists() else []
        pairs = []
        for s in s2_files:
            candidate_gt = self.gt_dir / f"{s.stem}.tif"
            if candidate_gt.exists():
                pairs.append((s, candidate_gt))
            else:
                prefix = s.stem.split('_')[0]
                cand = sorted(self.gt_dir.glob(f"{prefix}*.tif")) if self.gt_dir.exists() else []
                if cand:
                    pairs.append((s, cand[0]))
        if max_samples is not None:
            pairs = pairs[:max_samples]
        if not pairs:
            raise RuntimeError(f'No S2/GT pairs found in {self.s2_dir} / {self.gt_dir}')
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def _align_gt(self, gt_path, s2_shape, s2_transform, s2_crs):
        with rasterio.open(gt_path) as src_gt:
            gt = src_gt.read(1).astype('int32')
            src_transform = src_gt.transform
            src_crs = src_gt.crs
            gt_nodata = src_gt.nodata

        if gt.shape != s2_shape or src_crs != s2_crs:
            dst_fill = gt_nodata if gt_nodata is not None else 0
            dst = np.full(s2_shape, dst_fill, dtype=gt.dtype)
            reproject(
                source=gt,
                destination=dst,
                src_transform=src_transform,
                src_crs=src_crs,
                dst_transform=s2_transform,
                dst_crs=s2_crs,
                resampling=Resampling.nearest,
            )
            gt_aligned = dst
        else:
            gt_aligned = gt
        return gt_aligned, gt_nodata

    def __getitem__(self, idx):
        s2_path, gt_path = self.pairs[idx]
        with rasterio.open(s2_path) as src_s2:
            # Read selected bands
            bands = [src_s2.read(i).astype('float32') for i in self.band_idxs]
            s2_crs = src_s2.crs
            s2_transform = src_s2.transform
            height, width = src_s2.height, src_s2.width

        # Stack and scale to [0,1] like reflectance
        img = np.stack(bands, axis=0) / self.scale  # shape (C,H,W)

        # Align GT
        gt_aligned, gt_nodata = self._align_gt(gt_path, (height, width), s2_transform, s2_crs)
        # Map GT labels 1 and 3 to water=1 else 0
        gt_mask = np.isin(gt_aligned, [1, 3]).astype('uint8')

        # Optional simple augmentations
        if self.transforms is not None:
            img, gt_mask = self.transforms(img, gt_mask)

        # Convert to torch tensors
        img_t = torch.from_numpy(img).float()
        mask_t = torch.from_numpy(gt_mask).unsqueeze(0).float()  # (1,H,W)
        return img_t, mask_t


# Simple deterministic transforms (random flips)
def simple_transforms(img, mask, p_flip=0.5):
    # img: numpy (C,H,W), mask: (H,W)
    if random.random() < p_flip:
        # horizontal
        img = img[:, :, ::-1].copy()
        mask = mask[:, ::-1].copy()
    if random.random() < p_flip:
        # vertical
        img = img[:, ::-1, :].copy()
        mask = mask[::-1, :].copy()
    return img, mask


# Build dataset from the test split only (S2/GT pairs exist only in test/)
full_ds = WorldFloodsTilesDataset(DATA_ROOT, split='test', bands=('B4','B3','B8'), max_samples=None, transforms=simple_transforms)

# Deterministic train/val split from the available test pairs
from torch.utils.data import random_split

total = len(full_ds)
if total < 2:
    raise RuntimeError(f'Not enough samples in test split to create train/val sets (found {total})')

val_frac = 0.2
val_size = max(1, int(total * val_frac))
train_size = total - val_size

# Use a fixed seed for reproducibility
generator = torch.Generator().manual_seed(42)
train_ds, val_ds = random_split(full_ds, [train_size, val_size], generator=generator)

# DataLoaders (small batch size for laptop)
batch_size = 4
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)

print('Total samples (from test/):', total)
print('Train samples:', len(train_ds), 'Val samples:', len(val_ds))


# Training utilities: dice loss + bce+dice

def dice_loss_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    probs = probs.view(probs.size(0), -1)
    targets = targets.view(targets.size(0), -1)
    intersection = (probs * targets).sum(dim=1)
    union = probs.sum(dim=1) + targets.sum(dim=1)
    dice = (2 * intersection + eps) / (union + eps)
    return 1.0 - dice.mean()


def threshold_predictions(logits, thresh=0.5):
    return (torch.sigmoid(logits) >= thresh).to(torch.uint8)


# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# Build model
model = UNetResNet(encoder='resnet18', pretrained=False, out_channels=1, n_input_channels=3)
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

bce_loss = nn.BCEWithLogitsLoss()


# Small training loop (epochs configurable)
num_epochs = 3
for epoch in range(1, num_epochs + 1):
    model.train()
    running_loss = 0.0
    for imgs, masks in train_loader:
        imgs = imgs.to(device)
        masks = masks.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss_bce = bce_loss(logits, masks)
        loss_dice = dice_loss_logits(logits, masks)
        loss = loss_bce + loss_dice
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
    epoch_loss = running_loss / len(train_ds)

    # Validation
    model.eval()
    iou_list = []
    prec_list = []
    rec_list = []
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs = imgs.to(device)
            masks = masks.to(device)
            logits = model(imgs)
            preds = threshold_predictions(logits, thresh=0.5).float()

            # compute per-batch metrics
            preds_np = preds.cpu().numpy().astype(int)
            masks_np = masks.cpu().numpy().astype(int)
            for i in range(preds_np.shape[0]):
                y_pred = preds_np[i, 0].ravel()
                y_true = masks_np[i, 0].ravel()
                tp = int(((y_true == 1) & (y_pred == 1)).sum())
                fp = int(((y_true == 0) & (y_pred == 1)).sum())
                fn = int(((y_true == 1) & (y_pred == 0)).sum())
                prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
                rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
                inter = int(((y_true == 1) & (y_pred == 1)).sum())
                union = int(((y_true == 1) | (y_pred == 1)).sum())
                iou = inter / union if union > 0 else float('nan')
                prec_list.append(prec)
                rec_list.append(rec)
                iou_list.append(iou)

    mean_iou = np.nanmean(iou_list)
    mean_prec = np.nanmean(prec_list)
    mean_rec = np.nanmean(rec_list)

    print(f'Epoch {epoch}/{num_epochs} | train_loss={epoch_loss:.4f} | val IoU={mean_iou:.4f} | prec={mean_prec:.4f} | rec={mean_rec:.4f}')

# After training, save a checkpoint
ckpt_path = Path('unet_resnet18_checkpoint.pth')
torch.save({'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict()}, ckpt_path)
print('Saved checkpoint to', ckpt_path)

Total samples (from test/): 18
Train samples: 15 Val samples: 3
Using device: cpu


/Applications/anaconda3/envs/dsan6600/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Applications/anaconda3/envs/dsan6600/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


RuntimeError: Numpy is not available